# Collected data

Shows everything the collector has saved in `data/raw`. Use **Run All**; it pulls the newest
snapshots first.

In [ ]:
PULL_FIRST = True   # run `git pull` to get the newest snapshots

In [ ]:
import subprocess
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
DATA_DIR = Path.cwd() / "data" / "raw"
TZ = "America/Los_Angeles"

if PULL_FIRST:
    r = subprocess.run(["git", "pull", "--quiet"], capture_output=True, text=True)
    print(r.stdout or r.stderr or "git pull: up to date")

def load(source, kind):
    """All snapshots of one feed, oldest first. IDs stay text (e.g. route "51A")."""
    files = sorted(DATA_DIR.glob(f"*/*_{source}_{kind}.csv.gz"))
    frames = [pd.read_csv(f, dtype={"trip_id": str, "route_id": str, "vehicle_id": str,
                                    "stop_id": str, "start_date": str, "start_time": str})
              for f in files]
    frames = [f for f in frames if not f.empty]
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    # Readable Pacific time next to the unix timestamp we polled at.
    df.insert(1, "fetched_dt", pd.to_datetime(df["fetched_at"], unit="s", utc=True).dt.tz_convert(TZ))
    return df

data = {(s, k): load(s, k) for s in ("511", "act") for k in ("trips", "vehicles")}
print("loaded:", ", ".join(f"{s} {k}" for (s, k), df in data.items() if len(df)))

## Summary

In [ ]:
pd.DataFrame([
    {"source": s, "feed": k, "rows": len(df),
     "snapshots": df["fetched_at"].nunique() if len(df) else 0,
     "first (Pacific)": df["fetched_dt"].min() if len(df) else None,
     "last (Pacific)": df["fetched_dt"].max() if len(df) else None}
    for (s, k), df in data.items()
])

Snapshots per day (Pacific date):

In [ ]:
pd.DataFrame({f"{s} {k}": df.groupby(df["fetched_dt"].dt.date)["fetched_at"].nunique()
              for (s, k), df in data.items() if len(df)}).fillna(0).astype(int)

## Column reference

All times are Pacific. Unix timestamps are seconds since 1970 (UTC); `fetched_dt` is the
readable version of `fetched_at`. Codes like `0`, `1`, `2` are GTFS-Realtime enums, listed below.

### Trip updates (`trips_511`, `trips_act`)
One row per trip **per upcoming stop**: each snapshot keeps the next 5 predicted stops of every
active trip, so one trip appears in up to 5 rows per snapshot.

| Column | Meaning |
|---|---|
| `fetched_at` | When our collector polled the feed (unix). Same for every row of one snapshot. |
| `fetched_dt` | `fetched_at` as a Pacific date-time (added by this notebook). |
| `feed_ts` | When the agency generated the feed (unix). Usually a few seconds from `fetched_at`. |
| `trip_id` | Scheduled trip. Matches `trip_id` in the static GTFS schedule. |
| `route_id` | Route name, e.g. `51A`, `NL`, `72M`. |
| `direction_id` | `0` or `1`: the two directions of a route (meaning defined in the static schedule). |
| `start_date` | Service day of the trip, `YYYYMMDD`. |
| `start_time` | Scheduled start time of the trip, `HH:MM:SS`. Can pass `24:00` for trips after midnight. |
| `vehicle_id` | Bus number running the trip. |
| `stop_rank` | Position in the upcoming-stops list: `0` = the next stop, `1` = the one after, up to `4`. Added by our collector. |
| `stop_id` | Stop the prediction is for. **511 and AC Transit use different stop IDs** (e.g. `51096` vs `5224`). |
| `stop_sequence` | Order of this stop along the trip (1 = first stop). |
| `arrival_time` | Predicted arrival at the stop (unix). Missing for ~12% of rows, usually the first stop of a trip. |
| `departure_time` | Predicted departure (unix). Only filled for ~18% of rows, mostly first stops and timepoints. |
| `arrival_delay`, `departure_delay` | Seconds late vs. schedule. **Always empty in both feeds**; must be computed from the static schedule. |
| `schedule_relationship` | `0` = scheduled stop, `1` = stop will be skipped, `2` = no prediction, `3` = unscheduled. |

### Vehicle positions (`veh_511`, `veh_act`)
One row per bus per snapshot.

| Column | Meaning |
|---|---|
| `fetched_at`, `fetched_dt`, `feed_ts` | Same as above. |
| `vehicle_id` | Bus number. |
| `trip_id`, `route_id`, `start_date`, `start_time` | Trip the bus is running. Empty for ~2% of buses (not on a trip). |
| `direction_id` | `0` / `1` as above. **Always empty in the AC Transit feed**; use the 511 one or look up `trip_id`. |
| `lat`, `lon` | Position in degrees. |
| `bearing` | Heading in degrees clockwise from north (0 = north, 90 = east). |
| `speed` | Meters per second (multiply by 2.237 for mph). |
| `current_stop_sequence` | `stop_sequence` of the stop the bus is at or heading to. |
| `stop_id` | That stop's ID (different ID systems per source, as above). |
| `current_status` | `0` = about to arrive at the stop, `1` = stopped at it, `2` = in transit to it. |
| `timestamp` | When the bus reported this position (unix). Usually a few seconds to a minute before `feed_ts`. |
| `occupancy_status` | Crowding: `0` empty, `1` many seats, `2` few seats, `3` standing room only, `4` crushed standing, `5` full, `6` not accepting passengers, `7` no data. |

The two sources describe the same buses. They mostly agree, but stop IDs differ and each
feed is generated at a slightly different time (`feed_ts`).

## 511: trip updates

In [ ]:
trips_511 = data[("511", "trips")]
trips_511

## 511: vehicle positions

In [ ]:
veh_511 = data[("511", "vehicles")]
veh_511

## AC Transit API: trip updates

In [ ]:
trips_act = data[("act", "trips")]
trips_act

## AC Transit API: vehicle positions

In [ ]:
veh_act = data[("act", "vehicles")]
veh_act